In [209]:
import polars as pl

In [210]:
stores       = pl.read_csv("Full_Favorita_Dataset/stores.csv/stores.csv")
transactions = pl.read_csv("Full_Favorita_Dataset/transactions.csv/transactions.csv")
train        = pl.read_csv("Full_Favorita_Dataset/train.csv/train.csv",
                           schema_overrides={"unit_sales": pl.Float32,
                                             "onpromotion": pl.Boolean})

# Clip negatives
train = train.with_columns(pl.col("unit_sales").clip(lower_bound=0))

# Filter Type B
bc_stores = stores.filter(pl.col("type") == "B")

# Transaction stats
store_trans = (
    transactions
    .group_by("store_nbr")
    .agg(pl.col("transactions").mean().alias("avg_transactions"))
)

# Sales stats
store_sales = (
    train
    .group_by("store_nbr")
    .agg([
        pl.col("unit_sales").mean().alias("avg_sales"),
        pl.col("item_nbr").n_unique().alias("unique_items"),
    ])
)

# Join all
df_analysis = (
    bc_stores
    .join(store_trans, on="store_nbr", how="inner")
    .join(store_sales, on="store_nbr", how="inner")
)

# Normalized distance on both metrics
median_trans = df_analysis["avg_transactions"].median()
median_sales = df_analysis["avg_sales"].median()

best_stores = (
    df_analysis
    .with_columns([
        ((pl.col("avg_transactions") - median_trans).abs() / median_trans)
            .alias("trans_dist"),
        ((pl.col("avg_sales") - median_sales).abs() / median_sales)
            .alias("sales_dist"),
    ])
    .with_columns(
        (pl.col("trans_dist") + pl.col("sales_dist")).alias("total_distance")
    )
    .sort("total_distance")
)

print("Top 5 Type B Stores:")
print(best_stores.select([
    "store_nbr", "city", "type", "cluster",
    "avg_transactions", "avg_sales", "unique_items", "total_distance"
]).head(5))

best_store_nbr = best_stores[0, "store_nbr"]
print(f"\nRecommended Store: #{best_store_nbr}")

Top 5 Type B Stores:
shape: (5, 8)
┌───────────┬──────────┬──────┬─────────┬───────────────┬───────────┬───────────────┬──────────────┐
│ store_nbr ┆ city     ┆ type ┆ cluster ┆ avg_transacti ┆ avg_sales ┆ unique_items  ┆ total_distan │
│ ---       ┆ ---      ┆ ---  ┆ ---     ┆ ons           ┆ ---       ┆ ---           ┆ ce           │
│ i64       ┆ str      ┆ str  ┆ i64     ┆ ---           ┆ f32       ┆ u32           ┆ ---          │
│           ┆          ┆      ┆         ┆ f64           ┆           ┆               ┆ f64          │
╞═══════════╪══════════╪══════╪═════════╪═══════════════╪═══════════╪═══════════════╪══════════════╡
│ 39        ┆ Cuenca   ┆ B    ┆ 6       ┆ 1424.005959   ┆ 7.678902  ┆ 3245          ┆ 0.057932     │
│ 20        ┆ Quito    ┆ B    ┆ 6       ┆ 1564.949395   ┆ 7.756163  ┆ 3156          ┆ 0.068102     │
│ 31        ┆ Babahoyo ┆ B    ┆ 10      ┆ 1374.281287   ┆ 7.515152  ┆ 3109          ┆ 0.091204     │
│ 18        ┆ Quito    ┆ B    ┆ 16      ┆ 1322.655172   

In [211]:
import os
store_dataset_path = f"./Store{best_store_nbr}_Dataset"
os.makedirs(store_dataset_path, exist_ok=True)

In [212]:
train = pl.read_csv("Full_Favorita_Dataset/train.csv/train.csv")
train_store_39 = train.filter(pl.col('store_nbr') == best_store_nbr)
train_store_39 = train_store_39.drop(["id", "store_nbr"])
train_store_39.write_csv(store_dataset_path+"/train.csv")

In [213]:
train = pl.read_csv("Store39_Dataset/train.csv", try_parse_dates=True)
train.head()

date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2013-01-02,103501,6.0,null
2013-01-02,103665,6.0,null
2013-01-02,105576,5.0,null
2013-01-02,105693,12.0,null
2013-01-02,105857,6.0,null


In [214]:
from datetime import date
train = train.filter(pl.col('date') > date(2016, 6, 1))
train.head()

date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2016-06-02,103501,8.0,"""False"""
2016-06-02,103520,4.0,"""False"""
2016-06-02,103665,6.0,"""True"""
2016-06-02,105576,16.0,"""False"""
2016-06-02,105693,1.0,"""False"""


In [215]:
# Check null onpromotion by year
print("Null onpromotion by year:")
print(
    train
    .with_columns(pl.col("date").dt.year().alias("year"))
    .group_by("year")
    .agg([
        pl.col("onpromotion").null_count().alias("null_count"),
        pl.len().alias("total"),
    ])
    .with_columns(
        (pl.col("null_count") / pl.col("total") * 100)
        .round(2).alias("null_pct")
    )
    .sort("year")
)

Null onpromotion by year:
shape: (2, 4)
┌──────┬────────────┬────────┬──────────┐
│ year ┆ null_count ┆ total  ┆ null_pct │
│ ---  ┆ ---        ┆ ---    ┆ ---      │
│ i32  ┆ u32        ┆ u32    ┆ f64      │
╞══════╪════════════╪════════╪══════════╡
│ 2016 ┆ 0          ┆ 427305 ┆ 0.0      │
│ 2017 ┆ 0          ┆ 471728 ┆ 0.0      │
└──────┴────────────┴────────┴──────────┘


In [216]:
items = train.select("item_nbr").unique().to_numpy().flatten()
items.sort()
items

array([ 103501,  103520,  103665, ..., 2123410, 2123750, 2123839],
      shape=(3176,))

In [217]:
from datetime import timedelta, date
from tqdm import tqdm
delta = timedelta(days=1)

rows = []
for item in tqdm(items, desc="Items"):
    start_date = date(2016, 6, 1)
    end_date = date(2017, 8, 15)
    while start_date <= end_date:
        rows.append(
            {
                "date" : start_date,
                "item_nbr" : item,
                "unit_sales" : 0.0,
                "onpromotion" : "False"
            }
        )
        start_date += delta

zero_train = pl.DataFrame(rows)
zero_train.head()

Items: 100%|██████████| 3176/3176 [00:02<00:00, 1530.78it/s]


date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2016-06-01,103501,0.0,"""False"""
2016-06-02,103501,0.0,"""False"""
2016-06-03,103501,0.0,"""False"""
2016-06-04,103501,0.0,"""False"""
2016-06-05,103501,0.0,"""False"""


In [218]:
filled_train = pl.concat([zero_train, train]).unique(subset=["date", "item_nbr"], keep="last")
filled_train.head()

date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2017-06-13,1696007,0.0,"""False"""
2017-06-21,1088002,0.0,"""False"""
2016-08-11,1993337,0.0,"""False"""
2016-07-09,698642,18.0,"""False"""
2017-08-04,586824,0.0,"""False"""


In [219]:
sorted_filled_train = filled_train.sort(by=["item_nbr", "date"])
sorted_filled_train.head()

date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2016-06-01,103501,0.0,"""False"""
2016-06-02,103501,8.0,"""False"""
2016-06-03,103501,10.0,"""False"""
2016-06-04,103501,14.0,"""False"""
2016-06-05,103501,11.0,"""False"""


In [220]:
len(sorted_filled_train.filter((pl.col("item_nbr") == 2123839) & (pl.col("unit_sales") == 0.0)))

440

In [221]:
total_days = sorted_filled_train["date"].n_unique()
print(f"Total days: {total_days}")

item_activity = (
    sorted_filled_train
    .filter(pl.col("unit_sales") > 0)
    .group_by("item_nbr")
    .agg(pl.len().alias("days_with_sales"))
)

print("\nItems surviving by minimum activity rate:")
for pct in [0, 1, 5, 10, 15, 16, 17, 18, 19, 20, 25, 30, 40, 50]:
    min_days = int(total_days * pct / 100)
    n = item_activity.filter(
        pl.col("days_with_sales") >= min_days
    ).shape[0]
    total_items = item_activity.shape[0]
    print(f"   ≥ {pct:2d}% active ({min_days:4d} days): "
          f"{n:4d} items ({n/total_items*100:.1f}% kept)")

Total days: 441

Items surviving by minimum activity rate:
   ≥  0% active (   0 days): 3176 items (100.0% kept)
   ≥  1% active (   4 days): 3161 items (99.5% kept)
   ≥  5% active (  22 days): 3083 items (97.1% kept)
   ≥ 10% active (  44 days): 3003 items (94.6% kept)
   ≥ 15% active (  66 days): 2906 items (91.5% kept)
   ≥ 16% active (  70 days): 2892 items (91.1% kept)
   ≥ 17% active (  74 days): 2875 items (90.5% kept)
   ≥ 18% active (  79 days): 2860 items (90.1% kept)
   ≥ 19% active (  83 days): 2845 items (89.6% kept)
   ≥ 20% active (  88 days): 2816 items (88.7% kept)
   ≥ 25% active ( 110 days): 2731 items (86.0% kept)
   ≥ 30% active ( 132 days): 2632 items (82.9% kept)
   ≥ 40% active ( 176 days): 2401 items (75.6% kept)
   ≥ 50% active ( 220 days): 2213 items (69.7% kept)


In [222]:
total_days = sorted_filled_train["date"].n_unique()

item_activity = (
    sorted_filled_train
    .filter(pl.col("unit_sales") > 0)
    .group_by("item_nbr")
    .agg([
        pl.len().alias("days_with_sales"),
        pl.col("unit_sales").sum().alias("total_sales"),
    ])
    .with_columns(
        (pl.col("days_with_sales") / total_days * 100)
        .alias("activity_pct")
    )
)

item_activity.describe()

statistic,item_nbr,days_with_sales,total_sales,activity_pct
str,f64,f64,f64,f64
"""count""",3176.0,3176.0,3176.0,3176.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",1.2073e6,283.059824,2219.074302,64.185901
"""std""",580549.740671,128.911521,6122.170006,29.231638
"""min""",103501.0,1.0,0.313,0.226757
"""25%""",771156.0,181.0,511.0,41.043084
"""50%""",1.239862e6,317.0,1114.0,71.882086
"""75%""",1.673264e6,398.0,2301.0,90.249433
"""max""",2.123839e6,438.0,195541.0,99.319728


In [223]:
active_items = item_activity.filter(
    (pl.col("activity_pct") >= 5)
).select("item_nbr")

sorted_filled_train_filtered = sorted_filled_train.join(active_items, on="item_nbr", how="inner")

print(f"Items kept: {sorted_filled_train_filtered['item_nbr'].n_unique():,}")
print(f"Rows kept:  {len(sorted_filled_train_filtered):,}")

Items kept: 3,080
Rows kept:  1,358,280


######################################################

In [224]:
sales_data_deploy = train.join(active_items, on='item_nbr', how='inner')
sales_data_deploy = sales_data_deploy.sort(by=['item_nbr', 'date'])
sales_only_data_deploy= sales_data_deploy.select(['date', 'item_nbr', 'unit_sales'])
onpromotion = sales_data_deploy.select(['date', 'item_nbr', 'onpromotion'])
sales_data_deploy.head()

date,item_nbr,unit_sales,onpromotion
date,i64,f64,str
2016-06-02,103501,8.0,"""False"""
2016-06-03,103501,10.0,"""False"""
2016-06-04,103501,14.0,"""False"""
2016-06-05,103501,11.0,"""False"""
2016-06-06,103501,4.0,"""False"""


In [225]:
onpromotion.write_csv('Deployment_Data_Test/PromotionCalendar.csv')
sales_only_data_deploy.write_csv('Deployment_Data_Test/SalesData.csv')

################################################################

In [226]:
sorted_filled_train_filtered.write_csv("Store39_Dataset/sorted_filled_train.csv")

In [227]:
train = pl.read_csv("Store39_Dataset/sorted_filled_train.csv", try_parse_dates=True)

In [228]:
unique_items = train.select("item_nbr").unique().to_numpy().flatten()
unique_items

array([ 358519, 2010511,  765520, ..., 2001183,  915980,  736128],
      shape=(3080,))

In [229]:
items = pl.read_csv("Full_Favorita_Dataset/items.csv/items.csv")
items.head()

item_nbr,family,class,perishable
i64,str,i64,i64
96995,"""GROCERY I""",1093,0
99197,"""GROCERY I""",1067,0
103501,"""CLEANING""",3008,0
103520,"""GROCERY I""",1028,0
103665,"""BREAD/BAKERY""",2712,1


In [230]:
items_store39 = items.filter(pl.col("item_nbr").is_in(unique_items))
print(len(items_store39))
items_store39.head()

3080


item_nbr,family,class,perishable
i64,str,i64,i64
103501,"""CLEANING""",3008,0
103520,"""GROCERY I""",1028,0
103665,"""BREAD/BAKERY""",2712,1
105576,"""GROCERY I""",1045,0
105693,"""GROCERY I""",1034,0


In [231]:
# Check if sales are integer or float per item
float_items = (
    train
    .filter(pl.col("unit_sales") > 0)
    .with_columns(
        (pl.col("unit_sales") != pl.col("unit_sales").cast(pl.Int32).cast(pl.Float32))
        .alias("is_float")
    )
    .group_by("item_nbr")
    .agg(pl.col("is_float").any().alias("has_float_sales"))
)
float_items.head()

item_nbr,has_float_sales
i64,bool
1696047,true
2005274,false
1963324,false
679926,false
1913258,false


In [232]:
items_store39_check_float_sales = items_store39.join(float_items, on="item_nbr", how="inner")
items_store39_check_float_sales

item_nbr,family,class,perishable,has_float_sales
i64,str,i64,i64,bool
1696047,"""PRODUCE""",2032,1,true
2005274,"""BEVERAGES""",1122,0,false
1963324,"""PLAYERS AND ELECTRONICS""",5446,0,false
679926,"""EGGS""",2502,1,false
1913258,"""CLEANING""",3032,0,false
…,…,…,…,…
1209719,"""GROCERY I""",1042,0,false
979554,"""FROZEN FOODS""",2228,0,false
1584348,"""LADIESWEAR""",7780,0,false


In [233]:
items_store39_check_float_sales = items_store39_check_float_sales.sort(by="item_nbr")
items_store39_check_float_sales

item_nbr,family,class,perishable,has_float_sales
i64,str,i64,i64,bool
103501,"""CLEANING""",3008,0,false
103520,"""GROCERY I""",1028,0,false
103665,"""BREAD/BAKERY""",2712,1,false
105576,"""GROCERY I""",1045,0,false
105693,"""GROCERY I""",1034,0,false
…,…,…,…,…
2108569,"""GROCERY I""",1086,0,false
2110456,"""BEVERAGES""",1120,0,false
2112404,"""DAIRY""",2108,1,false


###########################################################

In [234]:
items_store39_check_float_sales.write_csv('Deployment_Data_Test/items.csv')

##################################################################

In [235]:
items_store39_check_float_sales.write_csv("Store39_Dataset/items.csv")

In [236]:
transactions = pl.read_csv("Full_Favorita_Dataset/transactions.csv/transactions.csv", try_parse_dates=True)
transactions = transactions.filter(pl.col(date) > date(2016, 6, 1))
transactions.head()

date,store_nbr,transactions
date,i64,i64
2016-06-02,1,1637
2016-06-02,2,1775
2016-06-02,3,2792
2016-06-02,4,1253
2016-06-02,5,1219


In [237]:
transactions_store39 = transactions.filter(pl.col("store_nbr") == 39).drop("store_nbr")
transactions_store39

date,transactions
date,i64
2016-06-02,1284
2016-06-03,1317
2016-06-04,1820
2016-06-05,1653
2016-06-06,1334
…,…
2017-08-11,1170
2017-08-12,1314
2017-08-13,1158


#############################################

In [238]:
transactions_store39.write_csv('Deployment_Data_Test/transactions.csv')

##################################################

In [239]:
from datetime import timedelta
delta = timedelta(days=1)

rows = []
start_date = date(2016, 6, 1)
end_date = date(2017, 8, 15)
while start_date <= end_date:
    rows.append({
        "date" : start_date,
        "transactions" : 0
    })
    start_date += delta
zero_transactions = pl.DataFrame(rows)
zero_transactions

date,transactions
date,i64
2016-06-01,0
2016-06-02,0
2016-06-03,0
2016-06-04,0
2016-06-05,0
…,…
2017-08-11,0
2017-08-12,0
2017-08-13,0


In [240]:
filled_transcations_store_39 = pl.concat([zero_transactions, transactions_store39]).unique(subset="date", keep="last").sort(by="date")
filled_transcations_store_39

date,transactions
date,i64
2016-06-01,0
2016-06-02,1284
2016-06-03,1317
2016-06-04,1820
2016-06-05,1653
…,…
2017-08-11,1170
2017-08-12,1314
2017-08-13,1158


In [241]:
filled_transcations_store_39.write_csv("Store39_Dataset/transactions.csv")

In [242]:
holidays = pl.read_csv("Full_Favorita_Dataset/holidays_events.csv/holidays_events.csv", try_parse_dates=True)
holidays.head()

date,type,locale,locale_name,description,transferred
date,str,str,str,str,bool
2012-03-02,"""Holiday""","""Local""","""Manta""","""Fundacion de Manta""",false
2012-04-01,"""Holiday""","""Regional""","""Cotopaxi""","""Provincializacion de Cotopaxi""",false
2012-04-12,"""Holiday""","""Local""","""Cuenca""","""Fundacion de Cuenca""",false
2012-04-14,"""Holiday""","""Local""","""Libertad""","""Cantonizacion de Libertad""",false
2012-04-21,"""Holiday""","""Local""","""Riobamba""","""Cantonizacion de Riobamba""",false


In [243]:
holidays_store39 = holidays.filter((pl.col("locale_name").is_in(["Ecuador", "Cuenca"])) & (pl.col("date") >= date(2016, 6, 1)) & (pl.col("date") <= date(2017, 8, 15)))
holidays_store39 = holidays_store39.drop(["locale_name", "description"])
holidays_store39 = holidays_store39.filter(pl.col('type') == 'Holiday')
holidays_store39.head()

date,type,locale,transferred
date,str,str,bool
2016-08-10,"""Holiday""","""National""",true
2016-10-09,"""Holiday""","""National""",false
2016-11-02,"""Holiday""","""National""",false
2016-11-03,"""Holiday""","""National""",false
2016-12-25,"""Holiday""","""National""",false


In [244]:
unique_types = holidays_store39.select("type").unique().to_numpy().flatten()
unique_types

array(['Holiday'], dtype=object)

In [245]:
unique_locales = holidays_store39.select("locale").unique().to_numpy().flatten()
unique_locales

array(['National', 'Local'], dtype=object)

In [246]:
holidays_store39 = holidays_store39.with_columns(
    [
        (pl.col("type") == htype).alias("is_"+htype) for htype in unique_types
    ]
)
holidays_store39.head()

date,type,locale,transferred,is_Holiday
date,str,str,bool,bool
2016-08-10,"""Holiday""","""National""",true,true
2016-10-09,"""Holiday""","""National""",false,true
2016-11-02,"""Holiday""","""National""",false,true
2016-11-03,"""Holiday""","""National""",false,true
2016-12-25,"""Holiday""","""National""",false,true


In [247]:
holidays_store39 = holidays_store39.with_columns(
    [
        (pl.col("locale") == hlocale).alias("is_"+hlocale) for hlocale in unique_locales
    ]
)
holidays_store39.head()

date,type,locale,transferred,is_Holiday,is_National,is_Local
date,str,str,bool,bool,bool,bool
2016-08-10,"""Holiday""","""National""",true,true,true,false
2016-10-09,"""Holiday""","""National""",false,true,true,false
2016-11-02,"""Holiday""","""National""",false,true,true,false
2016-11-03,"""Holiday""","""National""",false,true,true,false
2016-12-25,"""Holiday""","""National""",false,true,true,false


In [248]:
holidays_store39 = holidays_store39.drop(["type", "locale"])
holidays_store39.head()

date,transferred,is_Holiday,is_National,is_Local
date,bool,bool,bool,bool
2016-08-10,true,true,true,false
2016-10-09,false,true,true,false
2016-11-02,false,true,true,false
2016-11-03,false,true,true,false
2016-12-25,false,true,true,false


In [249]:
len(holidays_store39)

13

In [250]:
holidays_store39 = holidays_store39.group_by("date").agg(pl.col(pl.Boolean).any())
holidays_store39 = holidays_store39.sort(by="date")
holidays_store39.head()

date,transferred,is_Holiday,is_National,is_Local
date,bool,bool,bool,bool
2016-08-10,true,true,true,false
2016-10-09,false,true,true,false
2016-11-02,false,true,true,false
2016-11-03,false,true,true,false
2016-12-25,false,true,true,false


In [251]:
len(holidays_store39)

13

In [252]:
bool_cols = [c for c in holidays_store39.columns if holidays_store39[c].dtype == pl.Boolean and c!="transferred"]
bool_cols

['is_Holiday', 'is_National', 'is_Local']

In [253]:
holidays_store39 = holidays_store39.with_columns(
    [(~pl.col("transferred") & pl.col(col)).alias(col) for col in bool_cols]
)
holidays_store39.head()

date,transferred,is_Holiday,is_National,is_Local
date,bool,bool,bool,bool
2016-08-10,true,false,false,false
2016-10-09,false,true,true,false
2016-11-02,false,true,true,false
2016-11-03,false,true,true,false
2016-12-25,false,true,true,false


In [254]:
rows = []
cols = holidays_store39.columns
start_date = date(2016, 6, 1)
end_date = date(2017, 8, 15)
delta = timedelta(days=1)
while start_date <= end_date:
    dic = {
        "date" : start_date,
        "transferred" : False
    }
    for col in cols[2:]:
        dic[col] = False
    rows.append(dic)
    start_date += delta
zero_holidays = pl.DataFrame(rows)
zero_holidays.head()

date,transferred,is_Holiday,is_National,is_Local
date,bool,bool,bool,bool
2016-06-01,false,false,false,false
2016-06-02,false,false,false,false
2016-06-03,false,false,false,false
2016-06-04,false,false,false,false
2016-06-05,false,false,false,false


In [255]:
filled_holidays_store_39 = pl.concat([zero_holidays, holidays_store39]).unique(subset="date", keep="last").sort(by="date")
filled_holidays_store_39.head()

date,transferred,is_Holiday,is_National,is_Local
date,bool,bool,bool,bool
2016-06-01,false,false,false,false
2016-06-02,false,false,false,false
2016-06-03,false,false,false,false
2016-06-04,false,false,false,false
2016-06-05,false,false,false,false


In [256]:
filled_holidays_store_39 = filled_holidays_store_39.drop("transferred")
filled_holidays_store_39.head()

date,is_Holiday,is_National,is_Local
date,bool,bool,bool
2016-06-01,false,false,false
2016-06-02,false,false,false
2016-06-03,false,false,false
2016-06-04,false,false,false
2016-06-05,false,false,false


In [257]:
filled_holidays_store_39_closed = filled_holidays_store_39.with_columns(
    (
        ((pl.col("date").dt.month() == 1) & (pl.col("date").dt.day() == 1)) |
        ((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25))
    ).cast(pl.Int8).alias("is_likely_closed")
)
filled_holidays_store_39_closed.head()

date,is_Holiday,is_National,is_Local,is_likely_closed
date,bool,bool,bool,i8
2016-06-01,false,false,false,0
2016-06-02,false,false,false,0
2016-06-03,false,false,false,0
2016-06-04,false,false,false,0
2016-06-05,false,false,false,0


In [258]:
bool_cols = [c for c in filled_holidays_store_39_closed.columns 
             if filled_holidays_store_39_closed[c].dtype == pl.Boolean]

filled_holidays_store_39_closed = filled_holidays_store_39_closed.with_columns([
    pl.col(c).cast(pl.Int8) for c in bool_cols
])
filled_holidays_store_39_closed.head()

date,is_Holiday,is_National,is_Local,is_likely_closed
date,i8,i8,i8,i8
2016-06-01,0,0,0,0
2016-06-02,0,0,0,0
2016-06-03,0,0,0,0
2016-06-04,0,0,0,0
2016-06-05,0,0,0,0


############################################

In [259]:
filled_holidays_store_39_closed.write_csv('Deployment_Data_Test/holidays.csv')

##############################################

In [260]:
filled_holidays_store_39_closed.write_csv("Store39_Dataset/holidays.csv")

In [261]:
oil = pl.read_csv("Full_Favorita_Dataset/oil.csv/oil.csv", try_parse_dates=True)
oil.head()

date,dcoilwtico
date,f64
2013-01-01,null
2013-01-02,93.14
2013-01-03,92.97
2013-01-04,93.12
2013-01-07,93.2


In [262]:
from datetime import date
oil_store39 = oil.filter((pl.col("date") >= date(2016, 6, 1)) & (pl.col("date") <= date(2017, 8, 15)))
oil_store39 = oil_store39.rename({"dcoilwtico" : "oil_price"})
oil_store39.head()

date,oil_price
date,f64
2016-06-01,49.07
2016-06-02,49.14
2016-06-03,48.69
2016-06-06,49.71
2016-06-07,50.37


########################################

In [263]:
oil_store39.write_csv('Deployment_Data_Test/oil.csv')

#########################################

In [264]:
len(oil_store39)

315

In [265]:
from datetime import timedelta

rows = []
delta = timedelta(days=1)
start_date = date(2016, 6, 1)
end_date = date(2017, 8, 15)
while start_date <= end_date:
    rows.append(
        {
            "date" : start_date,
            "oil_price" : None
        }
    )
    start_date += delta
null_oil = pl.DataFrame(rows)
null_oil = null_oil.with_columns(pl.col("oil_price").cast(pl.Float64))
null_oil.head()

date,oil_price
date,f64
2016-06-01,null
2016-06-02,null
2016-06-03,null
2016-06-04,null
2016-06-05,null


In [266]:
len(null_oil)

441

In [267]:
filled_oil_store39 = pl.concat([null_oil, oil_store39]).unique(subset="date", keep="last").sort(by="date")
filled_oil_store39.head()

date,oil_price
date,f64
2016-06-01,49.07
2016-06-02,49.14
2016-06-03,48.69
2016-06-04,null
2016-06-05,null


In [268]:
filled_oil_store39 = filled_oil_store39.with_columns(pl.col("oil_price").forward_fill().backward_fill())
filled_oil_store39.head()

date,oil_price
date,f64
2016-06-01,49.07
2016-06-02,49.14
2016-06-03,48.69
2016-06-04,48.69
2016-06-05,48.69


In [269]:
filled_oil_store39.select("oil_price").null_count()

oil_price
u32
0


In [270]:
len(filled_oil_store39)

441

In [271]:
filled_oil_store39.write_csv("Store39_Dataset/oil.csv")

In [272]:
train = pl.read_csv("Store39_Dataset/sorted_filled_train.csv", try_parse_dates=True)
items = pl.read_csv("Store39_Dataset/items.csv")
transactions = pl.read_csv("Store39_Dataset/transactions.csv", try_parse_dates=True)
holidays = pl.read_csv("Store39_Dataset/holidays.csv", try_parse_dates=True)
oil = pl.read_csv("Store39_Dataset/oil.csv", try_parse_dates=True)

train = train.with_columns(pl.col("onpromotion").cast(pl.Int8))
items = items.with_columns(pl.col("has_float_sales").cast(pl.Int8))

master = train.join(items, on="item_nbr", how="left")

master = master.join(transactions, on="date", how="left")

master = master.join(holidays, on="date", how="left")

master = master.join(oil, on="date", how="left")

master.head()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price
date,i64,f64,i8,str,i64,i64,i8,i64,i64,i64,i64,i64,f64
2016-06-01,103501,0.0,0,"""CLEANING""",3008,0,0,0,0,0,0,0,49.07
2016-06-02,103501,8.0,0,"""CLEANING""",3008,0,0,1284,0,0,0,0,49.14
2016-06-03,103501,10.0,0,"""CLEANING""",3008,0,0,1317,0,0,0,0,48.69
2016-06-04,103501,14.0,0,"""CLEANING""",3008,0,0,1820,0,0,0,0,48.69
2016-06-05,103501,11.0,0,"""CLEANING""",3008,0,0,1653,0,0,0,0,48.69


In [273]:
master.shape

(1358280, 14)

In [274]:
master.null_count()

date,item_nbr,unit_sales,onpromotion,family,class,perishable,has_float_sales,transactions,is_Holiday,is_National,is_Local,is_likely_closed,oil_price
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [275]:
master = master.with_columns(
    pl.col('unit_sales').clip(lower_bound=0)
)

In [276]:
master.write_csv("Store39_Dataset/MasterDataset.csv")

In [277]:
MasterDatasetPreview = master.head(200)
MasterDatasetPreview.write_csv("Store39_Dataset/MasterDatasetPreview.csv")